Quantify drift with PSI and KS for features (e.g., log_amount, Time) and model scores (Test vs Stream1/2).

In [2]:
import numpy as np, pandas as pd
from pathlib import Path
from src.preprocessing import split_X_y, AmountTimeScaler
from src.utils import psi, ks_statistic
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

DATA_PROCESSED = Path("../data/processed")
test  = pd.read_csv(DATA_PROCESSED / "batch2_test.csv")
s1    = pd.read_csv(DATA_PROCESSED / "batch3_stream.csv")
s2    = pd.read_csv(DATA_PROCESSED / "batch4_stream.csv")


In [3]:
def add_log_amount(df):
    df = df.copy()
    df["log_amount"] = np.log1p(df["Amount"])
    return df

test_f = add_log_amount(test)
s1_f   = add_log_amount(s1)
s2_f   = add_log_amount(s2)

for feat in ["log_amount", "Time"]:
    psi_s1 = psi(test_f[feat], s1_f[feat], bins=10)
    psi_s2 = psi(test_f[feat], s2_f[feat], bins=10)
    print(f"PSI({feat}) Test->S1={psi_s1:.3f}, Test->S2={psi_s2:.3f}")


PSI(log_amount) Test->S1=0.073, Test->S2=0.116
PSI(Time) Test->S1=11.513, Test->S2=11.513


In [4]:
pipe = Pipeline([("scaler", AmountTimeScaler()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
X_tr, y_tr = split_X_y(pd.read_csv(DATA_PROCESSED/"batch1_train.csv"))
pipe.fit(X_tr, y_tr)

def scores(df):
    X, y = split_X_y(df)
    return pipe.predict_proba(X)[:,1], y

s_test, y_test = scores(test)
s_s1, y_s1     = scores(s1)
s_s2, y_s2     = scores(s2)

print("KS(score) Test vs S1:", ks_statistic(s_test, s_s1))
print("KS(score) Test vs S2:", ks_statistic(s_test, s_s2))


KS(score) Test vs S1: 0.16490000000000005
KS(score) Test vs S2: 0.24439999999999995
